In [21]:
# Sam Brown
# sam_brown@mines.edu
# Date: 06/0402925
# Goal: Observe how much each station is lifted by the tide

# Need full data for a year (2011) could be good.

import pandas as pd
import numpy as np
import datetime
import logging

import sys
sys.path.append("/Users/sambrown04/Documents/SURF/whillans-surf/notebooks/SURF")

import util.coordinate_transforms


In [23]:
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

def load_gps_data(file: str):
    """
    Load processed GPS data file into pandas DataFrame (no class version).
    
    Parameters:
        file - .pos precise point solution file
    
    Returns:
        data - DataFrame with columns: longitude, latitude, elevation, time,
               day_of_year, x, y, dist
        flip - Boolean flag for reversing order of readings
    """
    data = pd.DataFrame()
    flip = False
    
    try:
        # CSRS-PPP 2024 format
        d = pd.read_csv(file, skiprows=3, sep="\\s+")
        d = d.loc[
            d["HR:MN:SS.SS"].str.endswith("00.00")
            | d["HR:MN:SS.SS"].str.endswith("15.00")
            | d["HR:MN:SS.SS"].str.endswith("30.00")
            | d["HR:MN:SS.SS"].str.endswith("45.00")
            | d["HR:MN:SS.SS"].str.endswith("16.00")
            | d["HR:MN:SS.SS"].str.endswith("46.00")
        ]
        data["longitude"] = d["LONDD"] - d["LONMN"] / 60 - d["LONSS"] / 60 / 60
        data["latitude"] = d["LATDD"] - d["LATMN"] / 60 - d["LATSS"] / 60 / 60
        data["time"] = pd.to_datetime(d["YEAR-MM-DD"] + "T" + d["HR:MN:SS.SS"])
        data.loc[data["time"].dt.second == 46, "time"] -= pd.Timedelta(seconds=1)
        data.loc[data["time"].dt.second == 16, "time"] -= pd.Timedelta(seconds=1)
        data["day_of_year"] = d["DAYofYEAR"]
    except Exception as e1:
        try:
            logger.warning(f"Position file not from 2024 CSRS-PPP run: {e1}")
            d = pd.read_csv(file, skiprows=7, sep="\\s+")
            data["longitude"] = d["LON(d)"] - d["LON(m)"] / 60 - d["LON(s)"] / 60 / 60
            data["latitude"] = d["LAT(d)"] - d["LAT(m)"] / 60 - d["LAT(s)"] / 60 / 60
            data["time"] = pd.to_datetime(d["YEAR-MM-DD"] + "T" + d["HR:MN:SS.SSS"])
            data["day_of_year"] = d["DOY"]
        except Exception as e2:
            try:
                logger.warning(f"Trying alternate format: {e2}")
                d = pd.read_csv(file, skiprows=5, sep="\\s+")
                data["longitude"] = d["LONDD"] - d["LONMN"] / 60 - d["LONSS"] / 60 / 60
                data["latitude"] = d["LATDD"] - d["LATMN"] / 60 - d["LATSS"] / 60 / 60
                data["time"] = pd.to_datetime(d["YEAR-MM-DD"] + "T" + d["HR:MN:SS.SS"])
                data["day_of_year"] = d["DAYofYEAR"]
            except Exception as e3:
                logger.warning(f"Trying final alternate format: {e3}")
                d = pd.read_csv(file, skiprows=6, sep="\\s+")
                data["longitude"] = d["LON(d)"] - d["LON(m)"] / 60 - d["LON(s)"] / 60 / 60
                data["latitude"] = d["LAT(d)"] - d["LAT(m)"] / 60 - d["LAT(s)"] / 60 / 60
                data["time"] = pd.to_datetime(d["YEAR-MM-DD"] + "T" + d["HR:MN:SS.SSS"])
                data["day_of_year"] = d["DOY"]

    # Add elevation and other fields
    data["elevation"] = d["HGT(m)"]
    data["sats"] = d["NSV"]
    data["GDOP"] = d["GDOP"]

    # Convert to x/y coordinates
    x, y = util.coordinate_transforms.ll2xy(lon=data["longitude"], lat=data["latitude"])
    data["x"] = x
    data["y"] = y

    # Flip check
    if len(data.index) > 1:
        diff = data["time"].iloc[0] - data["time"].iloc[1]
        if diff > datetime.timedelta(seconds=0):
            flip = True
        
        x0 = data["x"].iloc[0]
        y0 = data["y"].iloc[0]
        data["dist"] = np.sqrt((data["x"] - x0) ** 2 + (data["y"] - y0) ** 2)

    return data, flip

In [37]:
file = "/Users/sambrown04/Documents/SURF/PPP/gz01/2011/gz010110.pos"
data, flag = load_gps_data(file)

In [41]:
data.head(50)

,longitude,latitude,time,day_of_year,elevation,sats,GDOP,x,y,dist
0,-163.669373,-84.148741,2011-01-11 23:59:45,11.999826,59.8001,11,1.8,-178910.252958,-610613.519691,0.000000
1,-163.669371,-84.148741,2011-01-11 23:59:30,11.999653,59.7818,11,1.8,-178910.276162,-610613.525819,0.024000
2,-163.669372,-84.148741,2011-01-11 23:59:15,11.999479,59.8076,11,1.8,-178910.266504,-610613.526757,0.015278
3,-163.669371,-84.148741,2011-01-11 23:59:00,11.999306,59.7939,11,1.8,-178910.276195,-610613.530539,0.025644
4,-163.669370,-84.148741,2011-01-11 23:58:45,11.999132,59.7906,11,1.8,-178910.284544,-610613.518003,0.031631
5,-163.669371,-84.148741,2011-01-11 23:58:30,11.998958,59.8060,11,1.8,-178910.278719,-610613.526646,0.026683
6,-163.669371,-84.148741,2011-01-11 23:58:15,11.998785,59.8109,11,1.8,-178910.279715,-610613.522256,0.026879
7,-163.669371,-84.148741,2011-01-11 23:58:00,11.998611,59.8356,11,1.8,-178910.280110,-610613.526239,0.027930
8,-163.669371,-84.148741,2011-01-11 23:57:45,11.998438,59.8248,11,1.8,-178910.276808,-610613.536034,0.028913
9,-163.669371,-84.148741,2011-01-11 23:57:30,11.998264,59.8270,11,1.8,-178910.280754,-610613.529203,0.029378
